# Script 1: Dataset Preprocessing
**SHDB-AF Preprocessing Pipeline | Steps 1 – 4**  
Master's Thesis: Deep Learning for Predicting Atrial Fibrillation

1. Set `DATA_ROOT` in the **USER SETTING** cell to the folder that contains the SHDB-AF records (`.dat/.hea/.atr`) and `AdditionalData.csv`.
2. Run all cells top to bottom (*Runtime → Run all*).
3. Step G may ask you to type a recording ID for unselected patients with more than one recording.

All outputs are written to `DATA_ROOT/outputs/preprocessing/`. The **CONFIGURATION** cell (parameters and Step 4 patient lists) reproduces the thesis settings and normally does not need to change.

In [ ]:
# -*- coding: utf-8 -*-
# =============================================================================
# Script 1: Dataset Preprocessing
# SHDB-AF Preprocessing Pipeline  |  Steps 1 – 4
# Master's Thesis: Deep Learning for Predicting Atrial Fibrillation
# =============================================================================
#
# PURPOSE:
#   Builds the train/test window index used in the thesis from the SHDB-AF
#   metadata CSV and rhythm annotation files, in four consecutive steps:
#
#   Step 1 — Filtering
#            Reads the metadata CSV and filters down to a clean patient pool.
#   Step 2 — AF-Free Region Extraction
#            Parses each patient's .atr rhythm annotation file and extracts
#            AF-free (Normal sinus rhythm) regions after applying safety buffers.
#   Step 3 — Window Slicing
#            Slices every AF-free region into fixed-duration windows of length
#            D seconds, with an optional overlap percentage P.
#   Step 4 — Train/Test Dataset Split
#            Splits the 40 selected patients into a training set and a test
#            set at the PATIENT level, and builds a secondary test set from the
#            patients that were not selected.
#
#   No ECG signal data is loaded or stored at any point. All outputs are
#   purely temporal index structures (start/end times in seconds).
#
# OUTPUT FILES (all written to DATA_ROOT/outputs/preprocessing/):
#   step1_output.json                            — filtering_log + usable_patients
#   step2_output.json                            — step_2_log + af_free_regions   (read by Step 3)
#   step3_output.json                            — window index                   (read by Step 4)
#   patient_window_summary.txt / .json / .pdf    — per-patient window table
#   recording_window_summary.txt / .json / .pdf  — per-recording window table
#   train_test_split.json                        — metadata + train set + test set
#   secondary_test_set.json                      — secondary test set from unselected patients
#
# DEPENDENCIES:
#   wfdb, pandas, numpy, reportlab (+ standard library).
#   reportlab is optional: it is only used for the two .pdf summaries. If it
#   is missing, the PDFs are skipped with a warning and everything else is
#   still written.
#
# ENVIRONMENT:
#   Google Colab or Jupyter. The script uses notebook-only syntax
#   (`%pip install`), so it will not run as a plain `python script.py`.
#   Google Drive is mounted automatically when DATA_ROOT points to it
#   (/content/drive/...); otherwise DATA_ROOT can be any local folder.
#
# USAGE:
#   1. Set DATA_ROOT in the USER SETTING cell to the folder that contains the
#      SHDB-AF records (.dat/.hea/.atr) and AdditionalData.csv. This is the
#      only required edit; the CONFIGURATION section holds the thesis
#      settings (parameters and Step 4 patient lists).
#   2. Run the script top to bottom.
#   3. Step 4 (Step G) asks you to choose a recording for any unselected
#      patient that has more than one; type the recording ID when prompted.
#
#   Each step reads the previous step's output back from disk
#   (step2_output.json → Step 3, step3_output.json → Step 4).
#
# =============================================================================

## USER SETTING

In [ ]:
# =============================================================================
# USER SETTING — the only line you need to edit
# =============================================================================
# Folder containing the SHDB-AF WFDB records (001.dat / 001.hea / 001.atr, ...)
# and AdditionalData.csv, exactly as downloaded from PhysioNet.
#   Colab example : "/content/drive/MyDrive/shdb-af/1.0.1"
#   Local example : "C:/data/shdb-af/1.0.1"  or  "/home/<user>/data/shdb-af/1.0.1"
#
# Every output of Scripts 1–3 is written under DATA_ROOT/outputs/.

DATA_ROOT = "/path/to/shdb-af/1.0.1"


## SETUP: Drive, Dependencies, Imports

In [ ]:
# 1. Mount Google Drive when DATA_ROOT points to it (Colab only)
if DATA_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount('/content/drive')

# 2. Install the medical waveform library (Steps 1–2) and the PDF library
#    used for the Step 3 summary PDFs
%pip install wfdb reportlab

import os
import json
import math
import random
from collections import OrderedDict

import numpy as np
import pandas as pd
import wfdb

# reportlab is only needed for the .pdf summaries. If the import fails, the
# PDFs are skipped with a warning, but step3_output.json and the .txt/.json
# summaries are still written, so a missing PDF library can never block Step 4.
try:
    from reportlab.lib.pagesizes import letter, landscape
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.platypus import SimpleDocTemplate, Preformatted
    REPORTLAB_AVAILABLE = True
except ImportError:
    REPORTLAB_AVAILABLE = False
    print("WARNING: reportlab is not installed. PDF summaries will be skipped. "
          "Run: %pip install reportlab")

## CONFIGURATION

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
# The file paths are derived from DATA_ROOT (USER SETTING cell). The settings
# below reproduce the thesis results; change them only to run a variant.

# ── Paths (derived from DATA_ROOT; identical in Scripts 1–3, do not edit) ─────
CSV_PATH            = os.path.join(DATA_ROOT, "AdditionalData.csv")
OUTPUT_ROOT         = os.path.join(DATA_ROOT, "outputs")
PREPROC_DIR         = os.path.join(OUTPUT_ROOT, "preprocessing")      # Script 1 outputs
SPLIT_FILE          = os.path.join(PREPROC_DIR, "train_test_split.json")
SECONDARY_TEST_FILE = os.path.join(PREPROC_DIR, "secondary_test_set.json")
CACHE_DIR           = os.path.join(OUTPUT_ROOT, "cache")              # Script 2 cache
EXPERIMENT_NAME     = "dilated_resnet_attention"
RESULTS_DIR         = os.path.join(OUTPUT_ROOT, "results", EXPERIMENT_NAME)   # Scripts 2–3

# ── Path checks — stop early with a clear message if a path is wrong ──────────
if not os.path.isdir(DATA_ROOT):
    raise FileNotFoundError(
        f"DATA_ROOT not found: {DATA_ROOT}\n"
        "Set DATA_ROOT in the USER SETTING cell to the SHDB-AF folder."
    )
if not os.path.isfile(CSV_PATH):
    raise FileNotFoundError(f"AdditionalData.csv not found in DATA_ROOT: {CSV_PATH}")
if not any(f.endswith(".atr") for f in os.listdir(DATA_ROOT)):
    raise FileNotFoundError(f"No .atr annotation files found in DATA_ROOT: {DATA_ROOT}")

# ── Seed ──────────────────────────────────────────────────────────────────────
RANDOM_SEED = 42


# ── Steps 1 & 2: Filtering and AF-Free Region Extraction ─────────────────────
#
# COLUMN NAMES: The defaults match the SHDB-AF spec naming conventions.
# Verify each name against your actual AdditionalData.csv header row and
# correct here if they differ. A clear assertion in Step 1 will report
# exactly which columns are missing if any name is wrong.
#
# AF TYPE LABELS: Verify that "PAF", "PerAF", "non_AF" match the exact
# string values in your AF_Type column.

CONFIG_STEP_1_2 = {

    # ── File paths ─────────────────────────────────────────────────────────
    "csv_path":   CSV_PATH,
    "wfdb_dir":   DATA_ROOT,    # folder with .dat/.hea/.atr files
    "output_dir": PREPROC_DIR,

    # ── CSV column names ───────────────────────────────────────────────────
    "col_subject_id":       "Subject_ID",
    "col_data_id":          "Data_ID",
    "col_af_type":          "AF_Type",
    "col_annotated":        "Annotated",
    "col_recording_length": "Holter_recording_length",   # expected format: HH:MM:SS

    # ── AF type label values ───────────────────────────────────────────────
    "label_paf":    "PAF",
    "label_per_af": "PerAF",
    "label_non_af": "non-AF",

    # ── Signal ────────────────────────────────────────────────────────────
    "sampling_rate": 200,   # Hz — SHDB-AF resampled sampling rate

    # ── Step 2: trim and buffer durations (in seconds) ────────────────────
    #
    # T: Recording edge trim — removed from the START and END of every
    #    recording (for both PAF and non-AF patients). Prevents edge
    #    artefacts from contaminating the usable window pool.
    #
    # X: Pre-AF buffer — removed from the END of any Normal (N) region
    #    that is immediately followed by an AFIB/AFL episode. Ensures the
    #    signal just before AF onset is excluded from the clean pool.
    #
    # Y: Post-AF buffer — removed from the START of any Normal (N) region
    #    that is immediately preceded by an AFIB/AFL episode. Ensures the
    #    signal immediately after AF termination is excluded.
    #
    # buffer_labels: Rhythm labels that trigger the X and Y buffers.
    #    Any N region adjacent to one of these labels gets buffered.
    "T": 300.0,
    "X": 300.0,
    "Y": 300.0,
    "buffer_labels": ["AFIB", "AFL"],

    "random_seed": RANDOM_SEED,

}


# ── Step 3: Window Slicing ────────────────────────────────────────────────────
#
# D : Window duration in seconds (float). Must be > 0.
#     Common values: 30.0, 60.0, 120.0
#
# P : Overlap percentage between consecutive windows (float). 0 <= P < 100.
#     P = 0.0  → no overlap; step = D          (non-overlapping baseline)
#     P = 50.0 → 50% overlap; step = D / 2
#     P = 75.0 → 75% overlap; step = D / 4
#
#     NOTE: P is expressed as a PERCENTAGE (0–100), not a fraction (0–1).

CONFIG_STEP_3 = {

    # ── File paths ─────────────────────────────────────────────────────────
    "input_path": os.path.join(PREPROC_DIR, "step2_output.json"),
    "output_dir": PREPROC_DIR,

    # ── Window slicing parameters ──────────────────────────────────────────
    "D": 60.0,    # Window duration in seconds. Must be > 0.
    "P":  0.0,    # Overlap percentage. 0 <= P < 100. (NOT a fraction: 50 = 50%)

    # ── Reproducibility ────────────────────────────────────────────────────
    "random_seed": RANDOM_SEED,
}


# ── Step 4: Train/Test Dataset Split ──────────────────────────────────────────

CONFIG_STEP_4 = {
    # ── File paths ─────────────────────────────────────────────────────────
    "input_file"  : os.path.join(PREPROC_DIR, "step3_output.json"),
    "output_file" : SPLIT_FILE,

    # ── TRAIN_RATIO ────────────────────────────────────────────────────────
    # Fraction of patients assigned to training. Applied independently within
    # PAF and non-AF groups to preserve class balance across both sets (C3).
    # Example: 0.8  →  16 PAF + 16 non-AF in train, 4 + 4 in test.
    "train_ratio" : 0.8,

    # ── WINDOW_CAP_MODE ────────────────────────────────────────────────────
    # Controls how many windows each patient contributes.
    #   "auto"     → cap = global minimum per-recording window count across
    #                all 40 patients (computed from filtered windows).
    #   "manual"   → cap = CONFIG_STEP_4["manual_cap"]  (must be set below)
    #   "uncapped" → no cap; all windows from the selected recording are used
    "window_cap_mode" : "auto",

    # ── MANUAL_CAP ─────────────────────────────────────────────────────────
    # Only used when window_cap_mode = "manual". Must be a positive integer.
    # Set to None when not in use.
    "manual_cap" : None,

    # ── RANDOM_SEED ────────────────────────────────────────────────────────
    # Seeds the patient-level shuffle (random.seed) so the identical
    # train/test split is reproduced on every run (constraint C4).
    "random_seed" : RANDOM_SEED,
}


# ── Step 4: Selected Patient Lists ────────────────────────────────────────────
# Each patient is mapped to their ONE selected recording ID. Some patients
# have multiple recordings in the raw dataset; only the recording listed here
# contributes windows to the dataset.
#
# Format: { "patient_id": "recording_id" }
# Both keys and values are strings to match step3_output.json field types.
#
# Rank and expected window counts (from the patient window summary in Step 3)
# are shown as comments for cross-reference.

PAF_PATIENTS = {
    "4899921": "20",   # rank  1 — 1,327 windows
    "4339581": "15",   # rank  2 — 1,323 windows
    "573723" : "35",   # rank  3 — 1,295 windows
    "5196228": "128",  # rank  4 — 1,216 windows
    "5793786": "129",  # rank  5 — 1,363 windows
    "5035872": "139",  # rank  6 — 1,418 windows
    "5113281": "108",  # rank  7 — 1,418 windows
    "5577891": "13",   # rank  8 — 1,418 windows
    "1393341": "110",  # rank  9 — 1,415 windows
    "5377233": "17",   # rank 10 — 1,392 windows
    "5494269": "50",   # rank 11 — 1,376 windows
    "5218251": "26",   # rank 12 — 1,374 windows
    "5305929": "39",   # rank 13 — 1,374 windows
    "5665512": "48",   # rank 14 — 1,370 windows
    "5101851": "28",   # rank 15 — 1,369 windows
    "76500"  : "12",   # rank 16 — 1,367 windows
    "5012646": "135",  # rank 17 — 1,343 windows
    "5041593": "23",   # rank 18 — 1,342 windows
    "1948335": "37",   # rank 19 — 1,339 windows
    "4615422": "105",  # rank 20 — 1,328 windows
}  # 20 PAF patients

NON_AF_PATIENTS = {
    "1289526": "54",   # rank 63 — 1,429 windows
    "1457796": "55",   # rank 64 — 1,429 windows
    "1805739": "56",   # rank 65 — 1,429 windows
    "2470860": "117",  # rank 66 — 1,429 windows
    "4282911": "124",  # rank 67 — 1,429 windows
    "4633584": "62",   # rank 68 — 1,429 windows
    "4789365": "103",  # rank 69 — 1,429 windows
    "5067291": "64",   # rank 70 — 1,429 windows
    "5126553": "65",   # rank 71 — 1,429 windows
    "5248590": "71",   # rank 72 — 1,429 windows
    "5423877": "77",   # rank 73 — 1,429 windows
    "5490792": "86",   # rank 74 — 1,429 windows
    "5624577": "84",   # rank 75 — 1,429 windows
    "5642820": "115",  # rank 76 — 1,429 windows
    "5686092": "116",  # rank 77 — 1,429 windows
    "5708133": "122",  # rank 78 — 1,429 windows
    "5133906": "118",  # rank 79 — 1,424 windows
    "5247120": "70",   # rank 80 — 1,424 windows
    "5317995": "73",   # rank 81 — 1,424 windows
    "1028352": "102",  # rank 82 — 1,401 windows
}  # 20 non-AF patients

# Expected window counts from the Step 3 patient window summary, used for
# cross-validation in Step A to confirm the JSON was generated correctly.

EXPECTED_WINDOW_COUNTS = {
    "4899921": 1327, "4339581": 1323, "573723": 1295,  "5196228": 1216,
    "5793786": 1363, "5035872": 1418, "5113281": 1418, "5577891": 1418,
    "1393341": 1415, "5377233": 1392, "5494269": 1376, "5218251": 1374,
    "5305929": 1374, "5665512": 1370, "5101851": 1369, "76500":   1367,
    "5012646": 1343, "5041593": 1342, "1948335": 1339, "4615422": 1328,

    "1289526": 1429, "1457796": 1429, "1805739": 1429, "2470860": 1429,
    "4282911": 1429, "4633584": 1429, "4789365": 1429, "5067291": 1429,
    "5126553": 1429, "5248590": 1429, "5423877": 1429, "5490792": 1429,
    "5624577": 1429, "5642820": 1429, "5686092": 1429, "5708133": 1429,
    "5133906": 1424, "5247120": 1424, "5317995": 1424, "1028352": 1401,
}

# ── Step 4 (Step G): Secondary test set ───────────────────────────────────────
# Add recording IDs (as strings) to exclude from the secondary test set entirely
MANUAL_DISCARD_RECORDINGS = []


# ── Configuration checks — catch a bad config before any work is done ────────

# Step 3 parameter validation
assert CONFIG_STEP_3["D"] > 0, \
    f"D must be > 0. Got D={CONFIG_STEP_3['D']}."
assert 0 <= CONFIG_STEP_3["P"] < 100, \
    f"P must satisfy 0 <= P < 100. Got P={CONFIG_STEP_3['P']}."

# Convenience structures derived from the Step 4 patient dicts above
ALL_PATIENTS       = {**PAF_PATIENTS, **NON_AF_PATIENTS}   # patient_id → recording_id
PAF_PATIENT_IDS    = list(PAF_PATIENTS.keys())
NON_AF_PATIENT_IDS = list(NON_AF_PATIENTS.keys())
ALL_SELECTED_IDS   = set(ALL_PATIENTS.keys())

# Sanity checks on the Step 4 patient list definitions themselves
assert len(PAF_PATIENTS)    == 20, "Expected exactly 20 PAF patients"
assert len(NON_AF_PATIENTS) == 20, "Expected exactly 20 non-AF patients"
assert len(ALL_SELECTED_IDS) == 40, (
    "Overlap detected between PAF and non-AF patient lists — check for duplicate IDs"
)
assert len(EXPECTED_WINDOW_COUNTS) == 40, "Expected 40 entries in EXPECTED_WINDOW_COUNTS"

## SHARED HELPERS

In [ ]:
# =============================================================================
# Shared Helpers
# =============================================================================

def set_global_seed(seed: int) -> None:
    """
    Set random seeds globally for reproducibility.

    Covers numpy and Python's built-in random module.

    PYTHONHASHSEED is also written to the environment, but this only affects
    Python processes started AFTER this call — it cannot change the hash
    randomisation of the interpreter that is already running. The pipeline
    does not rely on it: dict iteration order is insertion order (Python 3.7+),
    and every set of subject IDs is sorted before it is iterated.

    Steps 1–3 are fully deterministic (no random sampling is performed); all
    ordering is based on sorted keys, not random state. The only random
    operation is the Step 4 patient shuffle, which re-seeds random explicitly
    with CONFIG_STEP_4["random_seed"] immediately before shuffling.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def print_config(config: dict) -> None:
    """Print every key of a step's CONFIG except the file paths/directories."""
    print("=== CONFIG (non-path keys) ===")
    print(json.dumps(
        {k: v for k, v in config.items() if "path" not in k and "dir" not in k},
        indent=2
    ))


def save_json(data, filename: str, output_dir: str) -> None:
    """
    Serialise a dict (or OrderedDict) to a pretty-printed JSON file with
    2-space indentation. Prints the full saved path for confirmation.
    """
    path = os.path.join(output_dir, filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"  Saved → {path}")


set_global_seed(RANDOM_SEED)
os.makedirs(PREPROC_DIR, exist_ok=True)

## STEP 1 — Filtering

In [ ]:
# =============================================================================
# STEP 1: Filtering
# =============================================================================

def hms_to_seconds(hms: str) -> float:
    """
    Convert a recording duration string from HH:MM:SS format to total seconds.

    Example: "23:32:00" → 84720.0

    The Holter_recording_length column stores values in this format.
    Raises ValueError with a descriptive message if the format is unexpected,
    so bad rows are caught early rather than silently propagating wrong durations.
    """
    parts = str(hms).strip().split(":")
    if len(parts) != 3:
        raise ValueError(
            f"Unexpected recording length format: '{hms}'. "
            "Expected HH:MM:SS (e.g. '23:32:00')."
        )
    h, m, s = int(parts[0]), int(parts[1]), float(parts[2])
    return h * 3600.0 + m * 60.0 + s


def run_step1_filtering(config: dict) -> dict:
    """
    Step 1: Read AdditionalData.csv and produce the usable patient pool.

    Three sub-steps are applied in sequence:

      1.i   — Remove all PerAF patients entirely (they cannot provide
              clean AF-free training data).

      1.ii  — Remove unannotated recordings. Patients whose EVERY recording
              is unannotated are removed entirely (Case A). Patients with at
              least one annotated recording survive, but only their annotated
              recordings are kept (Case B).

      1.iii — Build the final usable_patients dictionary with recording
              durations converted from HH:MM:SS to seconds.

    Design note: the CSV is read with dtype=str throughout to prevent pandas
    from silently casting Data_ID integers (e.g. 1 → "1") in ways that could
    cause mismatches with the zero-padded filenames ("001"). String
    normalisation happens explicitly here and in _data_id_to_record_name().

    Returns
    -------
    dict with keys:
        filtering_log   — audit log of each removal sub-step
        usable_patients — {subject_id: {af_type, total_recordings, recordings}}
    """
    # Read all columns as strings to preserve Data_ID leading zeros
    df = pd.read_csv(config["csv_path"], dtype=str)

    # Verify required columns are present — catches wrong CONFIG names immediately
    required_cols = [
        config["col_subject_id"],
        config["col_data_id"],
        config["col_af_type"],
        config["col_annotated"],
        config["col_recording_length"],
    ]
    missing = [c for c in required_cols if c not in df.columns]
    assert not missing, (
        f"\n[Step 1] COLUMN MISMATCH — these columns from CONFIG were not found in the CSV:\n"
        f"  Missing: {missing}\n"
        f"  Actual CSV columns: {list(df.columns)}\n"
        f"Fix the col_* keys in CONFIG to match your CSV exactly."
    )

    # Strip leading/trailing whitespace from all relevant string columns
    for col in required_cols:
        df[col] = df[col].str.strip()

    # Parse the Annotated column — handles "True"/"False" strings or "1"/"0"
    df["_annotated_bool"] = df[config["col_annotated"]].str.lower().isin(
        ("true", "1", "yes")
    )

    filtering_log  = {}
    label_paf    = config["label_paf"]
    label_per_af = config["label_per_af"]
    label_non_af = config["label_non_af"]
    col_subj = config["col_subject_id"]
    col_data = config["col_data_id"]
    col_af   = config["col_af_type"]

    # ── Step 1.i: Remove PerAF patients ──────────────────────────────────────
    # Persistent AF patients are excluded entirely because their recordings
    # contain no meaningful AF-free baseline signal.
    per_af_mask = df[col_af] == label_per_af
    per_af_df   = df[per_af_mask].copy()
    df          = df[~per_af_mask].copy()

    removed_1i = {}
    for subj, grp in per_af_df.groupby(col_subj):
        removed_1i[str(subj)] = {
            "af_type":            label_per_af,
            "recordings_removed": sorted(grp[col_data].tolist()),
        }

    after_1i_total   = df[col_subj].nunique()
    after_1i_paf     = df[df[col_af] == label_paf][col_subj].nunique()
    after_1i_non_af  = df[df[col_af] == label_non_af][col_subj].nunique()

    filtering_log["step_1i_perAF_removed"] = {
        "patients_removed":   len(removed_1i),
        "recordings_removed": int(per_af_df.shape[0]),
        "removed_subjects":   removed_1i,
        "remaining_after_removal": {
            "total_patients":  after_1i_total,
            "PAF_patients":    after_1i_paf,
            "non_AF_patients": after_1i_non_af,
        },
    }

    # ── Step 1.ii: Remove unannotated recordings ──────────────────────────────
    # Only recordings with a .atr annotation file are usable for region extraction.
    annotated_df   = df[df["_annotated_bool"]].copy()
    unannotated_df = df[~df["_annotated_bool"]].copy()

    subjects_with_annotated   = set(annotated_df[col_subj].unique())
    subjects_with_unannotated = set(unannotated_df[col_subj].unique())

    # Case A: patient has NO annotated recording → remove entire patient
    fully_removed_subjects = subjects_with_unannotated - subjects_with_annotated
    # Case B: patient has at least one annotated recording → keep patient,
    #         drop only the unannotated recording(s)
    partially_affected_subjects = subjects_with_unannotated & subjects_with_annotated

    patients_fully_removed = {}
    for subj in sorted(fully_removed_subjects):
        af_type  = df.loc[df[col_subj] == subj, col_af].iloc[0]
        recs_all = sorted(df.loc[df[col_subj] == subj, col_data].tolist())
        patients_fully_removed[str(subj)] = {
            "af_type":            af_type,
            "recordings_removed": recs_all,
        }

    recordings_only_removed = {}
    count_recs_only   = 0
    count_recs_paf    = 0
    count_recs_non_af = 0
    for subj in sorted(partially_affected_subjects):
        af_type      = df.loc[df[col_subj] == subj, col_af].iloc[0]
        recs_dropped = sorted(
            unannotated_df.loc[unannotated_df[col_subj] == subj, col_data].tolist()
        )
        recordings_only_removed[str(subj)] = {
            "af_type":            af_type,
            "recordings_removed": recs_dropped,
        }
        n = len(recs_dropped)
        count_recs_only += n
        if af_type == label_paf:
            count_recs_paf += n
        else:
            count_recs_non_af += n

    fr_paf    = sum(1 for v in patients_fully_removed.values() if v["af_type"] == label_paf)
    fr_non_af = len(patients_fully_removed) - fr_paf

    filtering_log["step_1ii_no_atr_removed"] = {
        "patients_fully_removed": {
            "total_patients_removed":  len(patients_fully_removed),
            "PAF_patients_removed":    fr_paf,
            "non_AF_patients_removed": fr_non_af,
            "subjects":                patients_fully_removed,
        },
        "recordings_only_removed": {
            "total_recordings_removed":        count_recs_only,
            "recordings_from_PAF_patients":    count_recs_paf,
            "recordings_from_non_AF_patients": count_recs_non_af,
            "subjects":                        recordings_only_removed,
        },
    }

    # Retain only annotated recordings; remove fully-dropped patients
    df = annotated_df[~annotated_df[col_subj].isin(fully_removed_subjects)].copy()

    # ── Step 1.iii: Build usable_patients ─────────────────────────────────────
    total_patients  = df[col_subj].nunique()
    paf_patients    = df[df[col_af] == label_paf][col_subj].nunique()
    non_af_patients = df[df[col_af] == label_non_af][col_subj].nunique()
    total_recs      = df.shape[0]
    paf_recs        = df[df[col_af] == label_paf].shape[0]
    non_af_recs     = df[df[col_af] == label_non_af].shape[0]

    filtering_log["step_1iii_usable_pool_summary"] = {
        "total_patients":    total_patients,
        "PAF_patients":      paf_patients,
        "non_AF_patients":   non_af_patients,
        "total_recordings":  total_recs,
        "PAF_recordings":    paf_recs,
        "non_AF_recordings": non_af_recs,
    }

    usable_patients = {}
    for subj, grp in df.groupby(col_subj):
        af_type    = grp[col_af].iloc[0]
        recordings = {}
        for _, row in grp.iterrows():
            data_id      = str(row[col_data])
            duration_sec = hms_to_seconds(row[config["col_recording_length"]])
            recordings[data_id] = {"recording_duration_sec": duration_sec}
        usable_patients[str(subj)] = {
            "af_type":          af_type,
            "total_recordings": len(recordings),
            "recordings":       recordings,
        }

    assert len(usable_patients) == total_patients, (
        f"[Step 1] Patient count mismatch: usable_patients has {len(usable_patients)}, "
        f"expected {total_patients}."
    )

    step1_output = {
        "filtering_log":   filtering_log,
        "usable_patients": usable_patients,
    }

    print(f"[Step 1] Usable patients:   {total_patients} ({paf_patients} PAF, {non_af_patients} non-AF)")
    print(f"[Step 1] Usable recordings: {total_recs}")

    return step1_output

## STEP 2 — AF-Free Region Extraction

In [ ]:
# =============================================================================
# STEP 2: AF-Free Region Extraction
# =============================================================================

def _data_id_to_record_name(data_id: str) -> str:
    """
    Convert a Data_ID string to the zero-padded 3-digit record stem used
    on disk.

    SHDB-AF files are named 001.atr, 002.atr, ..., 128.atr.
    The CSV may store Data_ID as "1", "01", or "001" depending on how it
    was created. int() normalises all of these; zfill(3) pads to 3 digits.

    Examples:
      "1"   → "001"
      "12"  → "012"
      "104" → "104"
    """
    return str(int(data_id)).zfill(3)


def _parse_annotation_intervals(
    record_path: str,
    recording_duration_sec: float,
    sampling_rate: int,
) -> list:
    """
    Parse a WFDB .atr rhythm annotation file and return a list of
    (start_sec, end_sec, rhythm_label) tuples that cover the full recording.

    WFDB stores rhythm change markers in ann.aux_note with a leading '('
    character (e.g. '(AFIB', '(N', '(AFL'). Each entry marks the START
    of a new rhythm segment; the end of segment i is the start of segment
    i+1, or recording_duration_sec for the final segment.

    Cleaning applied to each aux_note entry:
      - Strip null bytes (\x00) — a common WFDB file artefact
      - Strip leading/trailing whitespace
      - Strip the leading '(' character

    Consecutive N-N annotations are preserved as separate segments (no
    merging). Each annotated rhythm segment is kept as an independent
    region so that the downstream windowing step has full visibility into
    the raw annotation structure.

    Parameters
    ----------
    record_path            : Full path to the record WITHOUT extension.
                             wfdb.rdann appends .atr automatically.
    recording_duration_sec : Duration from the CSV (Step 1), used as the
                             end time of the final rhythm segment.
    sampling_rate          : Samples per second (200 Hz for SHDB-AF).

    Returns
    -------
    List of (start_sec: float, end_sec: float, rhythm_label: str) tuples
    in chronological order. Empty list if no rhythm annotations are found.
    """
    ann = wfdb.rdann(record_path, 'atr')

    # Extract rhythm transitions — only entries with a non-empty label after cleaning
    transitions = []
    for sample, note in zip(ann.sample, ann.aux_note):
        clean = note.replace('\x00', '').strip().lstrip('(').strip()
        if clean:
            transitions.append((int(sample), clean))

    if not transitions:
        return []

    # Build (start_sec, end_sec, rhythm) intervals.
    # Each segment spans from its annotation sample to the next annotation
    # sample, or to the end of the recording for the final segment.
    intervals = []
    for i, (sample, rhythm) in enumerate(transitions):
        start_sec = sample / sampling_rate
        end_sec   = (transitions[i + 1][0] / sampling_rate
                     if i + 1 < len(transitions)
                     else recording_duration_sec)
        intervals.append((start_sec, end_sec, rhythm))

    return intervals


def _extract_n_regions(
    intervals: list,
    recording_usable_start: float,
    recording_usable_end: float,
    X: float,
    Y: float,
    buffer_labels: list,
) -> list:
    """
    Apply recording-edge trim boundaries and AF-adjacency safety buffers
    to all Normal (N) segments in the interval list and return the surviving
    usable regions.

    Key design decisions:

    - recording_usable_start and recording_usable_end are computed ONCE per
      recording by the caller (run_step2_af_free_regions) from the trim
      parameter T. This function never sees T directly.

    - The FULL interval list (including segments that fall inside the trim
      zone) is passed in. This is intentional: adjacency detection must be
      able to find buffer-label episodes that ended before the trim boundary
      so that the Y buffer is always measured from the actual AF→N transition
      time, not from T. Without the full list, a preceding AFIB segment that
      ended inside the trim zone would be invisible, and no Y buffer would be
      applied — leaving potentially contaminated signal.

    - X and Y buffers apply to ALL patients (PAF and non-AF). Patients with
      no buffer-label episodes in their recording are unaffected because the
      adjacency checks simply never evaluate as True for them.

    - Clamping: the trim boundary and the buffer boundary are applied
      independently; the more restrictive one wins:
          usable_start = max(recording_usable_start, seg_start + Y)
          usable_end   = min(recording_usable_end,   seg_end   - X)

    - Regions where usable_end <= usable_start after clamping are discarded.

    Parameters
    ----------
    intervals              : Full output of _parse_annotation_intervals,
                             including segments inside the trim zone.
    recording_usable_start : T seconds from recording start (pre-computed).
    recording_usable_end   : duration - T seconds (pre-computed).
    X                      : Pre-buffer-label buffer in seconds.
    Y                      : Post-buffer-label buffer in seconds.
    buffer_labels          : Rhythm labels that trigger X/Y buffers.

    Returns
    -------
    List of dicts with keys: start_sec, end_sec, duration_sec.
    In chronological order. Empty list if no region survives.
    """

    surviving = []

    for i, (seg_start, seg_end, rhythm) in enumerate(intervals):
        if rhythm != 'N':
            continue

        # Detect whether a buffered-label rhythm immediately borders this N region.
        # The full interval list (including inside the trim zone) is searched here
        # so that buffer-label segments near recording edges are also detected.
        preceded_by_buffered = (
            i > 0 and intervals[i - 1][2] in buffer_labels
        )
        followed_by_buffered = (
            i < len(intervals) - 1 and intervals[i + 1][2] in buffer_labels
        )

        # Compute usable_start: the more restrictive of the trim boundary
        # and the post-buffer-label Y margin wins (via max()).
        # Y is always measured from seg_start (the actual AF→N transition),
        # regardless of whether that transition falls inside the trim zone.
        usable_start = seg_start
        usable_start = max(usable_start, recording_usable_start)
        if preceded_by_buffered:
            usable_start = max(usable_start, seg_start + Y)

        # Compute usable_end: mirror of usable_start logic (via min()).
        usable_end = seg_end
        usable_end = min(usable_end, recording_usable_end)
        if followed_by_buffered:
            usable_end = min(usable_end, seg_end - X)

        # Discard zero- or negative-duration regions
        if usable_end <= usable_start:
            continue

        surviving.append({
            "start_sec":    round(usable_start, 6),
            "end_sec":      round(usable_end,   6),
            "duration_sec": round(usable_end - usable_start, 6),
        })

    # Chronological order is preserved because intervals are already sorted
    return surviving


def run_step2_af_free_regions(step1_output: dict, config: dict) -> dict:
    """
    Step 2: Parse .atr files for every patient and recording in usable_patients,
    apply the recording-edge trim and buffer-label safety margins, and build
    the af_free_regions dictionary of surviving usable Normal (N) regions.

    Four sub-steps:

      2.i   — Parse raw rhythm intervals per recording via
              _parse_annotation_intervals(). The full interval list,
              including segments inside the trim zone, is retained.

      2.ii  — Compute recording-level trim boundaries (recording_usable_start,
              recording_usable_end) once per recording from T, before extraction.

      2.iii — Apply trim boundaries and buffers via _extract_n_regions().
              Y is always measured from the actual AF→N transition, even if
              that transition falls inside the trim zone.

      2.iv  — Exclude recordings that yield zero usable regions, and exclude
              patients who end up with no surviving recordings.

    Returns
    -------
    dict with keys:
        step_2_log      — audit log of dropped recordings and excluded patients
        af_free_regions — {subject_id: {
                              af_type, total_usable_recordings,
                              total_usable_regions, total_usable_duration_sec,
                              recordings: {data_id: {
                                  usable_regions_count, usable_duration_sec,
                                  regions: {"region_1": {start_sec, end_sec,
                                                         duration_sec}, ...}
                              }}
                          }}
    """
    usable_patients = step1_output["usable_patients"]
    wfdb_dir      = config["wfdb_dir"]
    sr            = config["sampling_rate"]
    T             = config["T"]
    X             = config["X"]
    Y             = config["Y"]
    label_paf     = config["label_paf"]
    buffer_labels = config["buffer_labels"]

    af_free_regions = {}

    recordings_dropped_log = {}   # subject_id → {af_type, [dropped data_ids]}
    patients_excluded_log  = {}   # subject_id → {af_type, reason}

    for subject_id, patient_info in usable_patients.items():
        af_type        = patient_info["af_type"]
        surviving_recs = {}

        for data_id, rec_info in patient_info["recordings"].items():
            duration_sec = rec_info["recording_duration_sec"]
            record_name  = _data_id_to_record_name(data_id)
            record_path  = os.path.join(wfdb_dir, record_name)

            # Compute trim boundaries once per recording so that T is applied
            # at the recording level, not inside the per-region loop.
            recording_usable_start = T
            recording_usable_end   = duration_sec - T

            # Parse the annotation file. If it fails for any reason, treat the
            # recording as having no usable regions and drop it with a warning.
            # The full interval list (including inside the trim zone) is kept
            # so that adjacency detection in _extract_n_regions can look across
            # the trim boundary for buffer-label neighbours.
            try:
                intervals = _parse_annotation_intervals(record_path, duration_sec, sr)
            except Exception as e:
                print(
                    f"  [Step 2] WARNING: Failed to read annotation for "
                    f"subject={subject_id}, data_id={data_id} "
                    f"(record file: {record_name}): {e}. "
                    f"This recording will be dropped."
                )
                intervals = []

            regions = _extract_n_regions(
                intervals,
                recording_usable_start, recording_usable_end,
                X, Y, buffer_labels
            )

            if not regions:
                if subject_id not in recordings_dropped_log:
                    recordings_dropped_log[subject_id] = {
                        "af_type": af_type, "recordings_dropped": []
                    }
                recordings_dropped_log[subject_id]["recordings_dropped"].append(data_id)
                continue

            # Number regions sequentially from 1 within this recording
            region_dict = {}
            for k, region in enumerate(regions, start=1):
                region_dict[f"region_{k}"] = region

            rec_usable_dur = sum(r["duration_sec"] for r in regions)
            surviving_recs[data_id] = {
                "usable_regions_count": len(regions),
                "usable_duration_sec":  round(rec_usable_dur, 6),
                "regions":              region_dict,
            }

        # Exclude patients whose every recording was dropped
        if not surviving_recs:
            patients_excluded_log[subject_id] = {
                "af_type": af_type,
                "reason":  "no_usable_regions_after_trim_and_buffer",
            }
            continue

        total_regions  = sum(r["usable_regions_count"] for r in surviving_recs.values())
        total_duration = sum(r["usable_duration_sec"]   for r in surviving_recs.values())

        af_free_regions[subject_id] = {
            "af_type":                   af_type,
            "total_usable_recordings":   len(surviving_recs),
            "total_usable_regions":      total_regions,
            "total_usable_duration_sec": round(total_duration, 6),
            "recordings":                surviving_recs,
        }

    # ── Build step_2_log ──────────────────────────────────────────────────────
    total_recs_dropped  = sum(
        len(v["recordings_dropped"]) for v in recordings_dropped_log.values()
    )
    recs_dropped_paf    = sum(
        len(v["recordings_dropped"])
        for v in recordings_dropped_log.values()
        if v["af_type"] == label_paf
    )
    recs_dropped_non_af = total_recs_dropped - recs_dropped_paf

    pat_excl_paf    = sum(1 for v in patients_excluded_log.values() if v["af_type"] == label_paf)
    pat_excl_non_af = len(patients_excluded_log) - pat_excl_paf

    surv_patients  = len(af_free_regions)
    surv_paf       = sum(1 for v in af_free_regions.values() if v["af_type"] == label_paf)
    surv_non_af    = surv_patients - surv_paf
    surv_recs      = sum(v["total_usable_recordings"] for v in af_free_regions.values())
    surv_regions   = sum(v["total_usable_regions"]    for v in af_free_regions.values())
    surv_duration  = sum(v["total_usable_duration_sec"] for v in af_free_regions.values())

    step_2_log = {
        "recordings_dropped": {
            "total_recordings_dropped":        total_recs_dropped,
            "recordings_from_PAF_patients":    recs_dropped_paf,
            "recordings_from_non_AF_patients": recs_dropped_non_af,
            "subjects": {
                sid: {
                    "af_type":            v["af_type"],
                    "recordings_dropped": v["recordings_dropped"],
                }
                for sid, v in recordings_dropped_log.items()
            },
        },
        "patients_excluded": {
            "total_patients_excluded":  len(patients_excluded_log),
            "PAF_patients_excluded":    pat_excl_paf,
            "non_AF_patients_excluded": pat_excl_non_af,
            "subjects":                 patients_excluded_log,
        },
        "surviving_pool_summary": {
            "total_surviving_patients":     surv_patients,
            "PAF_patients":                 surv_paf,
            "non_AF_patients":              surv_non_af,
            "total_surviving_recordings":   surv_recs,
            "total_surviving_regions":      surv_regions,
            "total_surviving_duration_sec": round(surv_duration, 2),
        },
    }

    step2_output = {
        "step_2_log":      step_2_log,
        "af_free_regions": af_free_regions,
    }

    print(f"[Step 2] Surviving patients:   {surv_patients} ({surv_paf} PAF, {surv_non_af} non-AF)")
    print(f"[Step 2] Recordings dropped:   {total_recs_dropped}")
    print(f"[Step 2] Patients excluded:    {len(patients_excluded_log)}")

    return step2_output


def run_steps_1_and_2(config: dict) -> None:
    """
    Execute Steps 1 and 2 in sequence and save their outputs to JSON.

    Data flows linearly: Step 1 output is passed directly into Step 2.
    Both JSON files are saved to config["output_dir"] for use by Step 3.
    """
    sep = "=" * 60

    print(f"\n{sep}")
    print("SHDB-AF Preprocessing Pipeline  |  Steps 1 & 2")
    print(sep)

    print("\n── Step 1: Filtering ──────────────────────────────────────")
    step1_output = run_step1_filtering(config)
    save_json(step1_output, "step1_output.json", config["output_dir"])

    print("\n── Step 2: AF-Free Region Extraction ──────────────────────")
    step2_output = run_step2_af_free_regions(step1_output, config)
    save_json(step2_output, "step2_output.json", config["output_dir"])

    print(f"\n{sep}")
    print("Steps 1 & 2 complete. JSON index files saved.")
    print(sep)

## RUN STEPS 1 & 2

In [ ]:
print_config(CONFIG_STEP_1_2)
run_steps_1_and_2(CONFIG_STEP_1_2)

## STEP 3 — Window Slicing

In [ ]:
# =============================================================================
# STEP 3: Window Slicing
# =============================================================================
#
# Reads step2_output.json (usable AF-free regions per patient) and slices
# every region into fixed-duration windows of length D seconds, with an
# optional overlap percentage P. The output is a pure temporal index of
# (start_sec, end_sec) pairs anchored to each patient's original recording
# timeline.
#
# KEY DESIGN DECISIONS:
#   - Windows are generated in strict chronological order per patient:
#     recordings sorted by ascending numeric ID, regions by ascending index,
#     windows by ascending start_sec. This is a hard pipeline requirement.
#   - No partial windows: a window is only kept if its end falls within the
#     region boundary. Any trailing signal shorter than D is discarded.
#   - All (start_sec, end_sec) values are rounded to 3 decimal places.
#     No rounding is applied during intermediate computation.
#
# OUTPUT (all written to CONFIG_STEP_3["output_dir"]):
#   step3_output.json                           — window index        (read by Step 4)
#   patient_window_summary.txt / .json / .pdf   — per-patient table
#   recording_window_summary.txt / .json / .pdf — per-recording table (reference only)

def compute_step(D: float, P: float) -> float:
    """
    Compute the step size (advance) between consecutive window start positions.

    Formula:
        step = D * (1 - P / 100)

    Examples:
        D=60s, P=0%  → step=60.0s  (non-overlapping)
        D=60s, P=50% → step=30.0s  (50% overlap)
        D=60s, P=75% → step=15.0s  (75% overlap)

    Parameters
    ----------
    D : Window duration in seconds.
    P : Overlap percentage in range [0, 100).

    Returns
    -------
    float : Step size in seconds.
    """
    return D * (1.0 - P / 100.0)


def slice_region_into_windows(
    region_start: float,
    region_end: float,
    D: float,
    step: float,
    recording_id: str,
    region_id: str,
    global_index_offset: int,
) -> list:
    """
    Slice a single usable region into fixed-duration windows.

    A window starting at position i begins at:
        region_start + i * step

    A window is KEPT only if its end (start + D) falls within region_end.
    No partial windows are produced. The tail of the region that is shorter
    than D is silently discarded.

    The number of complete windows that fit is:
        floor((region_end - region_start - D) / step) + 1

    If region_end - region_start < D, this yields 0 windows.

    Floating-point note:
        No rounding is applied to intermediate boundary calculations.
        Final start_sec and end_sec are rounded to 3 decimal places.
        A tolerance of 1e-9 is used in the discard guard to absorb float
        drift and avoid dropping legitimately valid boundary-touching windows.

    Parameters
    ----------
    region_start         : Region start in seconds (from step2_output.json).
    region_end           : Region end in seconds (from step2_output.json).
    D                    : Window duration in seconds.
    step                 : Step between consecutive window starts (seconds).
    recording_id         : Recording ID string; stored in each window dict.
    region_id            : Region ID string (e.g. "region_1"); stored in each dict.
    global_index_offset  : window_index of the first window in this call,
                           continuing the patient-level counter from prior regions.

    Returns
    -------
    list : Ordered list of window dicts. Empty if the region is shorter than D.
    """
    region_duration = region_end - region_start

    # Region shorter than D — return empty; caller emits the warning
    if region_duration < D:
        return []

    n_windows = math.floor((region_duration - D) / step) + 1

    windows = []
    for i in range(n_windows):
        w_start = region_start + i * step
        w_end   = w_start + D

        # Safety guard: discard if floating-point drift pushes w_end over the boundary
        if w_end > region_end + 1e-9:
            break

        windows.append({
            "window_index": global_index_offset + i,
            "recording_id": recording_id,
            "region_id":    region_id,
            "start_sec":    round(w_start, 3),
            "end_sec":      round(w_end,   3),
        })

    return windows


def process_patient(
    patient_id: str,
    patient_data: dict,
    D: float,
    P: float,
    step: float,
) -> dict:
    """
    Generate all windows for one patient across all recordings and regions.

    Ordering (hard pipeline requirement):
        1. Recordings are iterated in ascending numeric order of recording_id.
           IDs are numeric strings; sorted by int() to avoid lexicographic
           ordering errors (e.g. "9" > "10" lexicographically but not
           numerically).
        2. Within each recording, regions are iterated in ascending numeric
           order of their integer suffix: region_1, region_2, etc.
        3. Within each region, windows are in ascending start_sec order,
           guaranteed by the loop structure in slice_region_into_windows.

    This produces a strictly chronological window list that reflects the
    temporal structure of the original 24-hour Holter recording.

    Parameters
    ----------
    patient_id   : Subject ID string (used in log messages only).
    patient_data : Entry from step2_output["af_free_regions"][patient_id].
    D            : Window duration in seconds.
    P            : Overlap percentage.
    step         : Precomputed step size in seconds.

    Returns
    -------
    dict : Patient-level output with af_type, total_windows, and windows list.
    """
    recordings = patient_data["recordings"]

    # Sort recordings by ascending numeric ID
    sorted_recording_ids = sorted(recordings.keys(), key=lambda x: int(x))

    all_windows = []
    global_window_counter = 0   # patient-level running window index

    for rec_id in sorted_recording_ids:
        rec_data = recordings[rec_id]
        regions  = rec_data["regions"]

        # Sort regions by their integer suffix: "region_1" → 1, "region_2" → 2
        # Integer sort avoids "region_10" < "region_2" string-comparison errors
        sorted_region_ids = sorted(
            regions.keys(),
            key=lambda r: int(r.split("_")[1])
        )

        for region_id in sorted_region_ids:
            region  = regions[region_id]
            r_start = region["start_sec"]
            r_end   = region["end_sec"]

            # Warn before calling the slicer so the message is informative
            # even though slice_region_into_windows silently returns [] for short regions
            if (r_end - r_start) < D:
                print(
                    f"  [Step 3] WARNING: Patient {patient_id}, "
                    f"recording {rec_id}, {region_id} — "
                    f"duration {r_end - r_start:.3f}s < D={D}s. "
                    f"Zero windows produced for this region."
                )

            windows = slice_region_into_windows(
                region_start=r_start,
                region_end=r_end,
                D=D,
                step=step,
                recording_id=rec_id,
                region_id=region_id,
                global_index_offset=global_window_counter,
            )

            all_windows.extend(windows)
            global_window_counter += len(windows)

    if global_window_counter == 0:
        print(
            f"  [Step 3] WARNING: Patient {patient_id} has ZERO total windows. "
            f"All regions are shorter than D={D}s. "
            f"Patient is included in output with empty windows list."
        )

    return {
        "af_type":             patient_data["af_type"],
        "total_windows":       global_window_counter,
        "window_duration_sec": float(D),
        "overlap_pct":         float(P),
        "step_sec":            round(step, 6),
        "windows":             all_windows,
    }


def sort_output(sliced_patients: dict) -> OrderedDict:
    """
    Sort the output dictionary:
        1. All PAF patients first.
        2. All non-AF patients second.
        3. Within each class, descending by total_windows (most windows first).

    OrderedDict is used to make the intended insertion order explicit.
    Standard Python 3.7+ dicts also preserve insertion order, but OrderedDict
    communicates intent clearly to future readers.

    Note: af_type values in step2_output.json are "PAF" and "non-AF" (hyphen).
    These are matched exactly here.
    """
    paf_patients    = {
        pid: v for pid, v in sliced_patients.items()
        if v["af_type"] == "PAF"
    }
    non_af_patients = {
        pid: v for pid, v in sliced_patients.items()
        if v["af_type"] == "non-AF"
    }

    # Guard: warn about any unexpected af_type values that would be silently dropped
    known_types = {"PAF", "non-AF"}
    all_types   = {v["af_type"] for v in sliced_patients.values()}
    unexpected  = all_types - known_types
    if unexpected:
        print(
            f"  [Step 3] WARNING: Unexpected af_type values found and excluded "
            f"from sort: {unexpected}. Check step2_output.json."
        )

    paf_sorted    = sorted(paf_patients.items(),    key=lambda x: x[1]["total_windows"], reverse=True)
    non_af_sorted = sorted(non_af_patients.items(), key=lambda x: x[1]["total_windows"], reverse=True)

    result = OrderedDict()
    for pid, data in paf_sorted + non_af_sorted:
        result[pid] = data

    return result


def run_step3_window_slicing(config: dict) -> OrderedDict:
    """
    Step 3 orchestrator: load Step 2 output, slice all patients into windows,
    validate the results, sort, and return.

    Pipeline:
        1. Load step2_output.json and extract af_free_regions.
        2. Validate input structure (required keys present per patient/recording/region).
        3. For each patient: generate windows via process_patient().
        4. Log summary statistics (window counts by class).
        5. Run three validation checks:
             (A) window_index is a contiguous 0-based sequence per patient.
             (B) Windows within the same region are in strictly ascending start_sec order.
             (C) end_sec - start_sec equals D for every window.
        6. Sort output (PAF first, then non-AF; descending by total_windows) and return.

    Parameters
    ----------
    config : CONFIG_STEP_3 dict. Must contain: D, P, input_path, output_dir.

    Returns
    -------
    OrderedDict ready for JSON serialisation.
    """
    D    = config["D"]
    P    = config["P"]
    step = compute_step(D, P)

    print(f"\n── Step 3: Window Slicing ──────────────────────────────────────")
    print(f"  D={D}s  |  P={P}%  |  step={step}s")

    # Load Step 2 output
    print(f"\n  Loading: {config['input_path']}")
    with open(config["input_path"], "r", encoding="utf-8") as f:
        step2_output = json.load(f)

    af_free_regions = step2_output["af_free_regions"]
    print(f"  Patients loaded from step2_output.json: {len(af_free_regions)}")

    # Validate input structure before doing any work.
    # Failing here is better than a KeyError mid-loop with no context.
    for pid, pdata in af_free_regions.items():
        assert "af_type"    in pdata,      f"Patient {pid} missing key 'af_type'."
        assert "recordings" in pdata,      f"Patient {pid} missing key 'recordings'."
        for rec_id, rec_data in pdata["recordings"].items():
            assert "regions" in rec_data,  f"Patient {pid}, recording {rec_id} missing key 'regions'."
            for reg_id, reg_data in rec_data["regions"].items():
                assert "start_sec" in reg_data, \
                    f"Patient {pid}, recording {rec_id}, {reg_id} missing 'start_sec'."
                assert "end_sec" in reg_data, \
                    f"Patient {pid}, recording {rec_id}, {reg_id} missing 'end_sec'."

    print("  Input structure validation: PASSED")

    # Process each patient
    sliced_patients      = {}
    total_windows_paf    = 0
    total_windows_non_af = 0
    zero_window_patients = []

    for patient_id, patient_data in af_free_regions.items():
        result = process_patient(patient_id, patient_data, D, P, step)
        sliced_patients[patient_id] = result

        if result["total_windows"] == 0:
            zero_window_patients.append(patient_id)

        if result["af_type"] == "PAF":
            total_windows_paf    += result["total_windows"]
        else:
            total_windows_non_af += result["total_windows"]

    # Summary
    n_paf    = sum(1 for v in sliced_patients.values() if v["af_type"] == "PAF")
    n_non_af = sum(1 for v in sliced_patients.values() if v["af_type"] == "non-AF")
    total_w  = total_windows_paf + total_windows_non_af

    print(f"\n  ── Summary ─────────────────────────────────────────────────")
    print(f"  Patients processed   : {len(sliced_patients)}  ({n_paf} PAF, {n_non_af} non-AF)")
    print(f"  PAF windows          : {total_windows_paf:,}")
    print(f"  non-AF windows       : {total_windows_non_af:,}")
    print(f"  Total windows        : {total_w:,}")
    print(f"  Class ratio (PAF/non): {total_windows_paf}/{total_windows_non_af}")

    if zero_window_patients:
        print(f"\n  WARNING — Patients with zero windows: {zero_window_patients}")
    else:
        print(f"  All {len(sliced_patients)} patients produced at least one window.")

    # Post-processing validation checks
    print(f"\n  ── Validation ──────────────────────────────────────────────")
    all_checks_passed = True

    for pid, pdata in sliced_patients.items():
        windows = pdata["windows"]

        if not windows:
            continue  # Patients with zero windows have nothing to validate

        # (A) window_index must be a contiguous 0-based sequence
        expected = list(range(len(windows)))
        actual   = [w["window_index"] for w in windows]
        if actual != expected:
            print(f"  FAIL (A) — Patient {pid}: window_index sequence broken. "
                  f"Expected 0..{len(windows)-1}, got {actual[:5]}...")
            all_checks_passed = False

        # (B) Windows within the same region must be in strictly ascending start_sec order
        for i in range(1, len(windows)):
            same_region = (
                windows[i]["recording_id"] == windows[i-1]["recording_id"]
                and windows[i]["region_id"] == windows[i-1]["region_id"]
            )
            if same_region and windows[i]["start_sec"] < windows[i-1]["start_sec"]:
                print(f"  FAIL (B) — Patient {pid}: window {i} start "
                      f"({windows[i]['start_sec']}) < window {i-1} start "
                      f"({windows[i-1]['start_sec']}) within same region.")
                all_checks_passed = False

        # (C) end_sec - start_sec must equal D for every window
        # Compare at 3 decimal places (matches the rounding applied during output)
        for w in windows:
            computed_dur = round(w["end_sec"] - w["start_sec"], 3)
            if abs(computed_dur - round(D, 3)) > 0.001:
                print(f"  FAIL (C) — Patient {pid}, window {w['window_index']}: "
                      f"end-start={computed_dur} != D={D}.")
                all_checks_passed = False

    if all_checks_passed:
        print("  PASS — All checks passed: index continuity, "
              "intra-region ordering, window duration consistency.")

    sorted_output = sort_output(sliced_patients)
    return sorted_output


def save_summary_files(summary_text: str, summary_json: dict, basename: str, output_dir: str) -> None:
    """
    Save one summary table to disk in three formats:
        <basename>.txt  — the console table, verbatim.
        <basename>.json — the machine-readable rows and totals.
        <basename>.pdf  — the console table typeset with reportlab
                          (landscape letter, monospaced Code style, 6 pt / 8 pt leading).

    Shared by both summaries (patient and recording) so they are always
    written the same way. If reportlab is not installed, the .pdf is skipped
    with a message; the .txt and .json are always written.

    Parameters
    ----------
    summary_text : The full table exactly as it was printed to the console.
    summary_json : Dict to serialise as the .json file.
    basename     : File name without extension, e.g. "patient_window_summary".
    output_dir   : Directory to write the three files into.
    """
    txt_path = os.path.join(output_dir, f"{basename}.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(summary_text + "\n")
    print(f"  Saved → {txt_path}")

    json_path = os.path.join(output_dir, f"{basename}.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(summary_json, f, indent=2, ensure_ascii=False)
    print(f"  Saved → {json_path}")

    if not REPORTLAB_AVAILABLE:
        print(f"  Skipped {basename}.pdf — reportlab is not installed.")
        return

    pdf_path = os.path.join(output_dir, f"{basename}.pdf")
    code_style = getSampleStyleSheet()["Code"]
    code_style.fontSize = 6
    code_style.leading = 8
    doc = SimpleDocTemplate(
        pdf_path,
        pagesize=landscape(letter),
        leftMargin=24, rightMargin=24, topMargin=24, bottomMargin=24,
    )
    doc.build([Preformatted(summary_text, code_style)])
    print(f"  Saved → {pdf_path}")


def print_patient_window_summary(step3_output: dict, output_dir: str = None) -> None:
    """
    Print a ranked, human-readable table of all patients and their window
    counts, broken down by recording.

    The ordering follows the step3_output JSON exactly (PAF first, then
    non-AF; descending by total_windows within each class). This is
    preserved automatically because step3_output is an OrderedDict.

    Output columns:
        Rank | Patient ID | AF Type | #Recs | Total Windows | Per-Recording Breakdown

    Per-recording breakdown is computed on the fly from the windows list
    so it always faithfully reflects the JSON content.

    The same summary is also saved to disk in three formats:
    patient_window_summary.txt, .json, and .pdf.

    IMPORTANT: Step 4 reads patient_window_summary.json and relies on its
    "patients" list and the "patient_id", "af_type" and "recording_breakdown"
    fields of each entry. Do not rename these without updating Step 4.

    Parameters
    ----------
    step3_output : OrderedDict returned by run_step3_window_slicing().
    output_dir : directory to save the summary files into. Defaults to
        CONFIG_STEP_3["output_dir"].
    """

    if output_dir is None:
        output_dir = CONFIG_STEP_3["output_dir"]

    lines = []

    def emit(text: str = "") -> None:
        print(text)
        lines.append(text)

    def get_recording_window_counts(windows: list) -> dict:
        """
        Count windows per recording_id from the flat windows list.
        Returns a dict {recording_id: count} in first-encountered (chronological) order.
        """
        counts = {}
        for w in windows:
            rec_id = w["recording_id"]
            counts[rec_id] = counts.get(rec_id, 0) + 1
        return counts

    COL_RANK  =  5
    COL_ID    = 12
    COL_TYPE  =  8
    COL_RECS  =  5
    COL_TOTAL = 14

    sep     = "=" * 85
    divider = "-" * 85
    header  = (
        f"{'Rank':>{COL_RANK}}  "
        f"{'Patient ID':<{COL_ID}}  "
        f"{'AF Type':<{COL_TYPE}}  "
        f"{'Recs':>{COL_RECS}}  "
        f"{'Total Windows':>{COL_TOTAL}}  "
        f"Per-Recording Breakdown"
    )

    emit(f"\n{sep}")
    emit("STEP 3 — Patient Window Summary  (PAF first → non-AF  |  desc. by total windows)")
    emit(sep)
    emit(header)
    emit(divider)

    prev_af_type = None
    summary_rows = []

    for rank, (patient_id, pdata) in enumerate(step3_output.items(), start=1):
        af_type       = pdata["af_type"]
        total_windows = pdata["total_windows"]
        windows       = pdata["windows"]

        # Insert a visual section break at the PAF → non-AF boundary
        if prev_af_type == "PAF" and af_type == "non-AF":
            emit(divider)
            emit(f"  {'— non-AF patients below —':^81}")
            emit(divider)
        prev_af_type = af_type

        rec_counts = get_recording_window_counts(windows)
        n_recs     = len(rec_counts)

        breakdown_parts = [f"[{rid}]: {cnt:,}" for rid, cnt in rec_counts.items()]
        breakdown_str   = "  ".join(breakdown_parts) if breakdown_parts else "—"

        row = (
            f"{rank:>{COL_RANK}}  "
            f"{patient_id:<{COL_ID}}  "
            f"{af_type:<{COL_TYPE}}  "
            f"{n_recs:>{COL_RECS}}  "
            f"{total_windows:>{COL_TOTAL},}  "
            f"{breakdown_str}"
        )
        emit(row)

        summary_rows.append({
            "rank": rank,
            "patient_id": patient_id,
            "af_type": af_type,
            "n_recordings": n_recs,
            "total_windows": total_windows,
            "recording_breakdown": rec_counts,
        })

    emit(divider)

    paf_patients    = {pid: v for pid, v in step3_output.items() if v["af_type"] == "PAF"}
    non_af_patients = {pid: v for pid, v in step3_output.items() if v["af_type"] == "non-AF"}

    total_paf_w    = sum(v["total_windows"] for v in paf_patients.values())
    total_non_af_w = sum(v["total_windows"] for v in non_af_patients.values())
    grand_total    = total_paf_w + total_non_af_w

    emit(
        f"{'':>{COL_RANK}}  "
        f"{'PAF  (' + str(len(paf_patients)) + ' patients)':<{COL_ID + COL_TYPE + COL_RECS + 4}}  "
        f"{total_paf_w:>{COL_TOTAL},}"
    )
    emit(
        f"{'':>{COL_RANK}}  "
        f"{'non-AF  (' + str(len(non_af_patients)) + ' patients)':<{COL_ID + COL_TYPE + COL_RECS + 4}}  "
        f"{total_non_af_w:>{COL_TOTAL},}"
    )
    emit(
        f"{'':>{COL_RANK}}  "
        f"{'TOTAL  (' + str(len(step3_output)) + ' patients)':<{COL_ID + COL_TYPE + COL_RECS + 4}}  "
        f"{grand_total:>{COL_TOTAL},}"
    )
    emit(sep)

    summary_json = {
        "patients": summary_rows,
        "totals": {
            "PAF":    {"patients": len(paf_patients),    "total_windows": total_paf_w},
            "non-AF": {"patients": len(non_af_patients), "total_windows": total_non_af_w},
            "grand_total_windows": grand_total,
        },
    }
    save_summary_files("\n".join(lines), summary_json, "patient_window_summary", output_dir)


def print_recording_window_summary(step3_output: dict, output_dir: str = None) -> None:
    """
    Print a ranked table with one row per RECORDING (not per patient), and
    flag recordings whose patient has windows in more than one recording.

    This is the recording-level view of the same data shown by
    print_patient_window_summary(). It is useful when building leakage-aware
    train/test splits: a patient with several recordings must never have one
    recording in train and another in test, and the Notes column lists every
    such patient's other recordings next to each row.

    Ordering:
        1. All PAF recordings first, then all non-AF recordings.
        2. Within each class, descending by window count.
        3. Ties keep their step3_output order (Python's sort is stable), i.e.
           the patient order of step3_output, then chronological recording order.

    Output columns:
        Rank | Rec ID | Patient ID | AF Type | Windows | Notes

    The same summary is also saved to disk in three formats:
    recording_window_summary.txt, .json, and .pdf.

    Note: recordings are discovered from the windows list of step3_output,
    so a recording that produced ZERO windows (all its regions shorter than D)
    does not appear here and is not counted in "Patient owns N recs".

    Parameters
    ----------
    step3_output : OrderedDict returned by run_step3_window_slicing().
    output_dir : directory to save the summary files into. Defaults to
        CONFIG_STEP_3["output_dir"].
    """

    if output_dir is None:
        output_dir = CONFIG_STEP_3["output_dir"]

    lines = []

    def emit(text: str = "") -> None:
        print(text)
        lines.append(text)

    # First pass: map each patient to the set of recording IDs that have windows
    patient_to_recs = {}
    for pid, pdata in step3_output.items():
        patient_to_recs[pid] = {w["recording_id"] for w in pdata["windows"]}

    # Second pass: one entry per (patient, recording) with its window count
    recordings_list = []
    for pid, pdata in step3_output.items():
        rec_counts = {}
        for w in pdata["windows"]:
            rid = w["recording_id"]
            rec_counts[rid] = rec_counts.get(rid, 0) + 1

        for rid, count in rec_counts.items():
            recordings_list.append({
                "recording_id":   rid,
                "patient_id":     pid,
                "af_type":        pdata["af_type"],
                "window_count":   count,
                "all_owner_recs": patient_to_recs[pid],
            })

    paf_recs    = [r for r in recordings_list if r["af_type"] == "PAF"]
    non_af_recs = [r for r in recordings_list if r["af_type"] == "non-AF"]

    paf_recs.sort(key=lambda x: x["window_count"], reverse=True)
    non_af_recs.sort(key=lambda x: x["window_count"], reverse=True)

    sorted_recs = paf_recs + non_af_recs

    COL_RANK =  5
    COL_REC  =  8
    COL_PAT  = 12
    COL_TYPE =  8
    COL_WIN  = 10

    sep     = "=" * 90
    divider = "-" * 90
    header  = (
        f"{'Rank':>{COL_RANK}}  "
        f"{'Rec ID':<{COL_REC}}  "
        f"{'Patient ID':<{COL_PAT}}  "
        f"{'AF Type':<{COL_TYPE}}  "
        f"{'Windows':>{COL_WIN}}  "
        f"Notes"
    )

    emit(f"\n{sep}")
    emit("STEP 3 — Individual Recording Summary  (PAF first → non-AF  |  desc. by windows)")
    emit(sep)
    emit(header)
    emit(divider)

    prev_af_type = None
    summary_rows = []

    for rank, rec in enumerate(sorted_recs, start=1):
        rid        = rec["recording_id"]
        pid        = rec["patient_id"]
        af_type    = rec["af_type"]
        windows    = rec["window_count"]
        all_recs   = rec["all_owner_recs"]
        total_recs = len(all_recs)

        # Insert a visual section break at the PAF → non-AF boundary
        if prev_af_type == "PAF" and af_type == "non-AF":
            emit(divider)
            emit(f"  {'— non-AF recordings below —':^86}")
            emit(divider)
        prev_af_type = af_type

        # Flag patients with several recordings and list the OTHER ones,
        # sorted numerically for a clean display
        notes      = ""
        other_recs = []
        if total_recs > 1:
            other_recs = sorted(all_recs - {rid}, key=lambda x: int(x))
            rec_label  = "Rec IDs" if len(other_recs) > 1 else "Rec ID"
            notes      = f"(Patient owns {total_recs} recs) Other {rec_label}: {', '.join(other_recs)}"

        row = (
            f"{rank:>{COL_RANK}}  "
            f"{rid:<{COL_REC}}  "
            f"{pid:<{COL_PAT}}  "
            f"{af_type:<{COL_TYPE}}  "
            f"{windows:>{COL_WIN},}  "
            f"{notes}"
        )
        emit(row)

        summary_rows.append({
            "rank":                 rank,
            "recording_id":         rid,
            "patient_id":           pid,
            "af_type":              af_type,
            "window_count":         windows,
            "is_multi_rec_patient": total_recs > 1,
            "patient_total_recs":   total_recs,
            "other_recording_ids":  other_recs,
        })

    emit(divider)

    total_paf_w    = sum(r["window_count"] for r in paf_recs)
    total_non_af_w = sum(r["window_count"] for r in non_af_recs)
    grand_total    = total_paf_w + total_non_af_w

    emit(
        f"{'':>{COL_RANK}}  "
        f"{'PAF  (' + str(len(paf_recs)) + ' recordings)':<{COL_REC + COL_PAT + COL_TYPE + 4}}  "
        f"{total_paf_w:>{COL_WIN},}"
    )
    emit(
        f"{'':>{COL_RANK}}  "
        f"{'non-AF  (' + str(len(non_af_recs)) + ' recordings)':<{COL_REC + COL_PAT + COL_TYPE + 4}}  "
        f"{total_non_af_w:>{COL_WIN},}"
    )
    emit(
        f"{'':>{COL_RANK}}  "
        f"{'TOTAL  (' + str(len(sorted_recs)) + ' recordings)':<{COL_REC + COL_PAT + COL_TYPE + 4}}  "
        f"{grand_total:>{COL_WIN},}"
    )
    emit(sep)

    summary_json = {
        "recordings": summary_rows,
        "totals": {
            "PAF":    {"recordings": len(paf_recs),    "total_windows": total_paf_w},
            "non-AF": {"recordings": len(non_af_recs), "total_windows": total_non_af_w},
            "grand_total_windows": grand_total,
        },
    }
    save_summary_files("\n".join(lines), summary_json, "recording_window_summary", output_dir)

## RUN STEP 3

In [ ]:
print_config(CONFIG_STEP_3)

step3_output = run_step3_window_slicing(CONFIG_STEP_3)
save_json(step3_output, "step3_output.json", CONFIG_STEP_3["output_dir"])

print_patient_window_summary(step3_output, CONFIG_STEP_3["output_dir"])
print_recording_window_summary(step3_output, CONFIG_STEP_3["output_dir"])

print("\n[Step 3] Complete.")

## STEP 4 — Train/Test Dataset Split

In [ ]:
# =============================================================================
# STEP 4: Train/Test Dataset Split
# =============================================================================
#
# Splits the curated pool of AF-free ECG windows into a training set and a
# test set. The split is performed at the PATIENT level — all windows from
# a single patient go to exactly one set.
#
# Additionally generates a secondary test set (secondary_test_set.json)
# from patients that were not included in the main 40-patient selection.
#
# INPUT:  step3_output.json  (produced by Step 3)
# OUTPUT:
#   train_test_split.json    —  metadata + train set + test set
#   secondary_test_set.json  —  secondary test set from unused patients
#
# KEY CONSTRAINTS:
#   C1 — Patient-level split: no patient appears in both train and test.
#   C2 — Temporal ordering: windows are taken from the chronological start
#         of each patient's recording; no shuffling of windows is applied.
#   C3 — Class balance: the split is applied independently within PAF and
#         non-AF groups to preserve class proportions in both sets.
#   C4 — Reproducibility: a fixed random seed controls the patient shuffle.
#   C5 — Output format: all per-patient metadata fields from Step 3 are
#         preserved in the output.
#   C6 — No data invention: only filtering and slicing; no window values
#         are modified, interpolated, or created.
#
# The Step 4 settings, patient lists and expected window counts are defined
# in the CONFIGURATION section at the top of this script.

print("Configuration loaded.")
print(f"  PAF patients defined    : {len(PAF_PATIENTS)}")
print(f"  Non-AF patients defined : {len(NON_AF_PATIENTS)}")
print(f"  Train ratio             : {CONFIG_STEP_4['train_ratio']}")
print(f"  Window cap mode         : {CONFIG_STEP_4['window_cap_mode']}")
print(f"  Manual cap              : {CONFIG_STEP_4['manual_cap']}")
print(f"  Random seed             : {CONFIG_STEP_4['random_seed']}")

## Step 4 · Step A — Load and Filter

In [ ]:
# =============================================================================
# Step A — Load and Filter
# =============================================================================
# Load the full window pool from step3_output.json, retain only the 40 selected
# patients, and verify that every expected patient ID is present with the
# correct af_type label.

print(f"Loading: {CONFIG_STEP_4['input_file']}")

if not os.path.exists(CONFIG_STEP_4["input_file"]):
    raise FileNotFoundError(
        f"Input file not found: {CONFIG_STEP_4['input_file']}\n"
        "Verify the file path in CONFIG_STEP_4 is correct."
    )

with open(CONFIG_STEP_4["input_file"], "r", encoding="utf-8") as f:
    pool = json.load(f)

print(f"  Total patients in file   : {len(pool)}")

# Verify all 40 selected patient IDs are present. Stop immediately if any are missing.
missing_ids = [pid for pid in ALL_SELECTED_IDS if pid not in pool]
if missing_ids:
    raise ValueError(
        f"ERROR: {len(missing_ids)} patient ID(s) not found in the input JSON:\n"
        f"  Missing: {sorted(missing_ids)}\n"
        "Verify that step3_output.json was generated from the correct dataset."
    )
print(f"  All 40 selected IDs found in file.  ✓")

# Filter: retain only the 40 selected patients
filtered_pool = {pid: pool[pid] for pid in ALL_SELECTED_IDS}
assert len(filtered_pool) == 40, (
    f"Expected 40 patients after filtering, got {len(filtered_pool)}"
)

# Verify af_type labels match our patient dicts (C6 — no data invention)
label_errors = []
for pid in PAF_PATIENT_IDS:
    actual = filtered_pool[pid].get("af_type")
    if actual != "PAF":
        label_errors.append(f"  Patient {pid}: expected 'PAF',    got '{actual}'")
for pid in NON_AF_PATIENT_IDS:
    actual = filtered_pool[pid].get("af_type")
    if actual != "non-AF":
        label_errors.append(f"  Patient {pid}: expected 'non-AF', got '{actual}'")

if label_errors:
    raise ValueError(
        f"ERROR: af_type label mismatch for {len(label_errors)} patient(s):\n"
        + "\n".join(label_errors)
        + "\nCheck that PAF_PATIENTS / NON_AF_PATIENTS match the JSON labels."
    )
print(f"  af_type labels match patient dicts.  ✓")

# Build per-patient window counts restricted to each patient's designated recording.
# step3_output.json may contain windows from multiple recordings per patient;
# we must restrict all downstream processing to the one recording listed in
# ALL_PATIENTS so window counts and caps are computed correctly.
recording_window_counts = {}
recording_check_errors  = []

for pid in ALL_PATIENTS:
    target_rec  = ALL_PATIENTS[pid]
    all_windows = filtered_pool[pid]["windows"]

    rec_windows = [w for w in all_windows if w["recording_id"] == target_rec]
    n = len(rec_windows)
    recording_window_counts[pid] = n

    expected = EXPECTED_WINDOW_COUNTS[pid]
    if n != expected:
        recording_check_errors.append(
            f"  Patient {pid} (rec {target_rec}): "
            f"expected {expected} windows, found {n} in JSON"
        )

if recording_check_errors:
    print(f"WARNING: window count mismatch for {len(recording_check_errors)} patient(s):")
    for msg in recording_check_errors:
        print(msg)
    print("  Proceeding, but verify step3_output.json is correct.")
else:
    print(f"  Per-recording window counts match expected counts for all 40 patients.  ✓")

min_pid = min(recording_window_counts, key=recording_window_counts.get)
max_pid = max(recording_window_counts, key=recording_window_counts.get)
print(f"  Window count range (selected recording) :")
print(f"    Min : {recording_window_counts[min_pid]} "
      f"(patient {min_pid}, rec {ALL_PATIENTS[min_pid]}, {filtered_pool[min_pid]['af_type']})")
print(f"    Max : {recording_window_counts[max_pid]} "
      f"(patient {max_pid}, rec {ALL_PATIENTS[max_pid]}, {filtered_pool[max_pid]['af_type']})")


def print_unselected_recordings_summary(full_pool: dict, selected_patients_dict: dict) -> None:
    """
    Print a table of all recordings in step3_output.json that were NOT selected
    for Step 4. Groups by AF type (PAF first, then non-AF) and sorts descending
    by window count.

    This is a diagnostic/auditing function. It has no effect on output data.

    Parameters
    ----------
    full_pool              : Complete step3_output.json dict (all patients).
    selected_patients_dict : ALL_PATIENTS dict (patient_id → selected_recording_id).
    """
    unselected_items = []

    for pid, pdata in full_pool.items():
        af_type     = pdata.get("af_type", "Unknown")
        windows     = pdata.get("windows", [])
        unique_recs = set(w["recording_id"] for w in windows)
        total_recs  = len(unique_recs)

        is_patient_selected        = pid in selected_patients_dict
        selected_rec_for_patient   = selected_patients_dict.get(pid)

        for rid in unique_recs:
            # Skip the recording that was actually selected — we only want unselected ones
            if is_patient_selected and rid == selected_rec_for_patient:
                continue

            if is_patient_selected:
                notes = f"(Patient owns {total_recs} recs) Rec ID [{selected_rec_for_patient}] was selected."
            else:
                if total_recs > 1:
                    other_recs = sorted(list(unique_recs - {rid}), key=lambda x: int(x))
                    notes = (
                        f"(Patient owns {total_recs} recs) Patient excluded entirely. "
                        f"Other unselected: {', '.join(other_recs)}"
                    )
                else:
                    notes = "Patient excluded entirely."

            rec_window_count = sum(1 for w in windows if w["recording_id"] == rid)

            unselected_items.append({
                "patient_id":   pid,
                "recording_id": rid,
                "af_type":      af_type,
                "window_count": rec_window_count,
                "notes":        notes,
            })

    # Sort: PAF first (0), then non-AF (1); descending by window count within each class
    unselected_items.sort(key=lambda x: (
        0 if x["af_type"] == "PAF" else 1,
        -x["window_count"]
    ))

    COL_PAT, COL_REC, COL_TYPE, COL_WIN = 12, 8, 8, 9
    sep     = "=" * 105
    divider = "-" * 105
    header  = (
        f"{'Patient ID':<{COL_PAT}}  "
        f"{'Rec ID':<{COL_REC}}  "
        f"{'AF Type':<{COL_TYPE}}  "
        f"{'Windows':>{COL_WIN}}  "
        f"Notes"
    )

    print(f"\n{sep}")
    print("STEP 4 - Unselected Data Summary (PAF first -> non-AF | desc. by windows)")
    print(sep)
    print(header)
    print(divider)

    prev_af_type = None

    if not unselected_items:
        print("All data in step3_output.json was selected. Nothing omitted.")
    else:
        for item in unselected_items:
            if prev_af_type == "PAF" and item["af_type"] == "non-AF":
                print(divider)
                print(f"  {'- non-AF unselected recordings below -':^100}")
                print(divider)
            prev_af_type = item["af_type"]

            row = (
                f"{item['patient_id']:<{COL_PAT}}  "
                f"{item['recording_id']:<{COL_REC}}  "
                f"{item['af_type']:<{COL_TYPE}}  "
                f"{item['window_count']:>{COL_WIN},}  "
                f"{item['notes']}"
            )
            print(row)

    print(divider)
    print(f"Total Unselected Recordings: {len(unselected_items)}")
    print(sep)


print_unselected_recordings_summary(pool, ALL_PATIENTS)

## Step 4 · Step B — Compute the Window Cap

In [ ]:
# =============================================================================
# Step B — Compute the Window Cap
# =============================================================================
# Determine how many windows each patient will contribute to the dataset.
#
# The cap is computed from recording_window_counts — the number of windows in
# each patient's selected recording only, not the total_windows field in the
# JSON (which may span multiple recordings).

mode = CONFIG_STEP_4["window_cap_mode"]

if mode == "auto":
    # Cap = minimum per-recording window count across all 40 patients.
    cap     = min(recording_window_counts.values())
    min_pid = min(recording_window_counts, key=recording_window_counts.get)
    print(f"Window cap mode  : auto")
    print(f"Cap value        : {cap}  (minimum found at patient '{min_pid}', "
          f"rec {ALL_PATIENTS[min_pid]}, {filtered_pool[min_pid]['af_type']})")

elif mode == "manual":
    if not isinstance(CONFIG_STEP_4["manual_cap"], int) or CONFIG_STEP_4["manual_cap"] <= 0:
        raise ValueError(
            "WINDOW_CAP_MODE is 'manual' but CONFIG_STEP_4['manual_cap'] is not a positive integer. "
            f"Got: {CONFIG_STEP_4['manual_cap']!r}. Set a valid value before re-running."
        )
    cap = CONFIG_STEP_4["manual_cap"]
    insufficient = [
        (pid, recording_window_counts[pid])
        for pid in recording_window_counts
        if recording_window_counts[pid] < cap
    ]
    if insufficient:
        print(f"  WARNING: {len(insufficient)} patient(s) have fewer windows "
              f"than the manual cap ({cap}):")
        for pid, n in insufficient:
            print(f"    Patient {pid} (rec {ALL_PATIENTS[pid]}): {n} windows available")
    print(f"Window cap mode  : manual")
    print(f"Cap value        : {cap}")

elif mode == "uncapped":
    cap = None
    print(f"Window cap mode  : uncapped")
    print(f"Cap value        : None  (each patient contributes all windows "
          f"from their selected recording)")

else:
    raise ValueError(
        f"Invalid WINDOW_CAP_MODE: '{mode}'. "
        "Must be one of: 'auto', 'manual', 'uncapped'."
    )

## Step 4 · Step C — Patient-Level Train/Test Split

In [ ]:
# =============================================================================
# Step C — Patient-Level Train/Test Split
# =============================================================================
# Patients are split at the patient level (C1). The split is applied
# independently within PAF and non-AF groups so both sets contain a
# proportional mix of each class (C3). A fixed RANDOM_SEED ensures the same
# split on every run (C4).

paf_ids   = list(PAF_PATIENT_IDS)
nonaf_ids = list(NON_AF_PATIENT_IDS)

# Shuffle each group independently using RANDOM_SEED.
# Implementation note: random.seed() is called once before both shuffles.
# The non-AF shuffle continues from the RNG state left by the PAF shuffle.
# This is fully deterministic (same seed → same output always) and is the
# standard practice for this type of split. Do NOT call random.seed() a
# second time between the two shuffles — that would make them identical.
random.seed(CONFIG_STEP_4["random_seed"])
random.shuffle(paf_ids)
random.shuffle(nonaf_ids)

# Compute split sizes using floor division to guarantee we never exceed count
n_train_paf   = math.floor(len(paf_ids)   * CONFIG_STEP_4["train_ratio"])
n_train_nonaf = math.floor(len(nonaf_ids) * CONFIG_STEP_4["train_ratio"])
n_test_paf    = len(paf_ids)   - n_train_paf
n_test_nonaf  = len(nonaf_ids) - n_train_nonaf

train_paf_ids   = paf_ids[:n_train_paf]
test_paf_ids    = paf_ids[n_train_paf:]
train_nonaf_ids = nonaf_ids[:n_train_nonaf]
test_nonaf_ids  = nonaf_ids[n_train_nonaf:]

train_patient_ids = train_paf_ids + train_nonaf_ids
test_patient_ids  = test_paf_ids  + test_nonaf_ids

# Integrity checks — all must pass before continuing
overlap = set(train_patient_ids) & set(test_patient_ids)
assert len(overlap) == 0, (
    f"LEAKAGE DETECTED: {len(overlap)} patient(s) appear in both train and test.\n"
    f"  Overlapping IDs: {overlap}"
)

total_assigned = len(train_patient_ids) + len(test_patient_ids)
assert total_assigned == 40, (
    f"Expected 40 patients assigned in total, got {total_assigned}. "
    "Some patients may have been dropped."
)

assert len(train_paf_ids)   == n_train_paf
assert len(test_paf_ids)    == n_test_paf
assert len(train_nonaf_ids) == n_train_nonaf
assert len(test_nonaf_ids)  == n_test_nonaf

print("Patient split complete.")
print(f"  Train : {n_train_paf} PAF + {n_train_nonaf} non-AF = {len(train_patient_ids)} patients")
print(f"  Test  : {n_test_paf} PAF  + {n_test_nonaf}  non-AF = {len(test_patient_ids)} patients")
print(f"  No patient leakage.  ✓")
print(f"\n  Train PAF IDs    : {sorted(train_paf_ids)}")
print(f"  Train non-AF IDs : {sorted(train_nonaf_ids)}")
print(f"  Test  PAF IDs    : {sorted(test_paf_ids)}")
print(f"  Test  non-AF IDs : {sorted(test_nonaf_ids)}")

## Step 4 · Step D — Window Selection per Patient

In [ ]:
# =============================================================================
# Step D — Window Selection per Patient
# =============================================================================
# Window selection is a two-stage process:
#
#   Stage 1 — Filter by recording ID: keep only windows whose recording_id
#             matches the designated recording from ALL_PATIENTS. Windows from
#             any other recording in step3_output.json are discarded here.
#
#   Stage 2 — Apply the cap: take the first `cap` windows from the filtered,
#             chronologically ordered list (C2). This selects the earliest
#             consecutive windows. No shuffling or reordering is ever applied.

def select_windows_for_patient(patient_data: dict, recording_id: str, cap) -> dict:
    """
    Select windows for a single patient from their designated recording,
    applying the window cap.

    Constraints enforced:
      C2 — Temporal ordering: windows are taken from the start of the filtered
           chronological list (filtered[0:cap]). No shuffling is applied.
      C5 — Output format: all original per-patient metadata fields are preserved.
      C6 — No data invention: only filtering and slicing; no window values
           are modified, interpolated, or created.

    Parameters
    ----------
    patient_data (dict) : One patient entry from step3_output.json.
    recording_id (str)  : The recording ID to retain (from ALL_PATIENTS).
    cap (int | None)    : Max windows to select. None = all filtered windows.

    Returns
    -------
    dict : Output entry preserving input structure, with 'windows' restricted
           to the target recording and truncated to cap, and 'total_windows'
           updated to reflect the actual count.
    """
    all_windows = patient_data["windows"]

    # Stage 1 — Filter to the designated recording only
    rec_windows = [w for w in all_windows if w["recording_id"] == recording_id]

    if len(rec_windows) == 0:
        raise ValueError(
            f"No windows found for recording_id='{recording_id}' "
            f"in patient with af_type='{patient_data.get('af_type')}'. "
            "Check that ALL_PATIENTS recording IDs match the JSON data."
        )

    # Stage 2 — C2: Take the first `cap` windows (earliest in chronological order).
    # rec_windows is already sorted by window_index (guaranteed by step3_output.json).
    # Slicing [:cap] is safe even if cap > len(rec_windows).
    selected = rec_windows[:cap] if cap is not None else rec_windows

    entry = {
        "af_type"            : patient_data["af_type"],
        "recording_id"       : recording_id,
        "total_windows"      : len(selected),
        "window_duration_sec": patient_data["window_duration_sec"],
        "overlap_pct"        : patient_data["overlap_pct"],
        "step_sec"           : patient_data["step_sec"],
        "windows"            : selected,
    }

    # Verify window_index is strictly increasing within the selection (C2)
    if len(selected) > 1:
        indices = [w["window_index"] for w in selected]
        is_strictly_increasing = all(
            indices[i] < indices[i + 1] for i in range(len(indices) - 1)
        )
        assert is_strictly_increasing, (
            f"TEMPORAL ORDERING VIOLATED: window_index is not strictly increasing "
            f"for recording '{recording_id}' of patient "
            f"'{patient_data.get('af_type', '?')}'. "
            "Verify that step3_output.json windows are pre-sorted by window_index."
        )

    return entry


# Apply window selection to train and test patients
train_data = {}
for pid in train_patient_ids:
    train_data[pid] = select_windows_for_patient(
        patient_data = filtered_pool[pid],
        recording_id = ALL_PATIENTS[pid],
        cap          = cap,
    )

test_data = {}
for pid in test_patient_ids:
    test_data[pid] = select_windows_for_patient(
        patient_data = filtered_pool[pid],
        recording_id = ALL_PATIENTS[pid],
        cap          = cap,
    )

all_processed = {**train_data, **test_data}
print(f"Window selection complete.")
print(f"  Train patients processed : {len(train_data)}")
print(f"  Test  patients processed : {len(test_data)}")

if cap is not None:
    at_cap    = sum(1 for d in all_processed.values() if d["total_windows"] == cap)
    below_cap = [
        (pid, all_processed[pid]["total_windows"])
        for pid in all_processed if all_processed[pid]["total_windows"] < cap
    ]
    print(f"  Patients at exactly cap  : {at_cap} / 40")
    if below_cap:
        print(f"  Patients below cap       : {len(below_cap)}")
        for pid, n in below_cap:
            print(f"    {pid}: {n} windows (cap = {cap})")
    else:
        print(f"  All 40 patients have exactly {cap} windows.  ✓")
else:
    total_all = sum(d["total_windows"] for d in all_processed.values())
    print(f"  Total windows selected   : {total_all}  (uncapped)")

## Step 4 · Steps E & F — Assemble Output Structure and Save

In [ ]:
# =============================================================================
# Steps E & F — Assemble Output Structure and Save
# =============================================================================

total_train_windows = sum(train_data[pid]["total_windows"] for pid in train_data)
total_test_windows  = sum(test_data[pid]["total_windows"]  for pid in test_data)

output = {
    "metadata": {
        "train_ratio"         : CONFIG_STEP_4["train_ratio"],
        "window_cap_mode"     : CONFIG_STEP_4["window_cap_mode"],
        "window_cap"          : cap,
        "random_seed"         : CONFIG_STEP_4["random_seed"],
        "n_train_paf"         : n_train_paf,
        "n_train_nonaf"       : n_train_nonaf,
        "n_test_paf"          : n_test_paf,
        "n_test_nonaf"        : n_test_nonaf,
        "total_train_windows" : total_train_windows,
        "total_test_windows"  : total_test_windows,
    },
    "train": train_data,
    "test" : test_data,
}

step4_output_dir = os.path.dirname(CONFIG_STEP_4["output_file"])
if step4_output_dir and not os.path.exists(step4_output_dir):
    os.makedirs(step4_output_dir, exist_ok=True)

with open(CONFIG_STEP_4["output_file"], "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

file_size_mb = os.path.getsize(CONFIG_STEP_4["output_file"]) / (1024 * 1024)
print(f"Output saved to : {CONFIG_STEP_4['output_file']}")
print(f"File size       : {file_size_mb:.2f} MB")

## Step 4 · Output Verification Report

In [ ]:
# =============================================================================
# Output Verification Report
# =============================================================================
# Re-reads the saved file from disk and runs all integrity checks.
# All checks must show ✓ before passing this output to the training pipeline.

with open(CONFIG_STEP_4["output_file"], "r", encoding="utf-8") as f:
    saved = json.load(f)

SEP = "=" * 70
print(SEP)
print("OUTPUT VERIFICATION REPORT")
print("Step 4 — Train/Test Dataset Split")
print(SEP)

meta        = saved["metadata"]
saved_train = saved["train"]
saved_test  = saved["test"]

# Patient counts
total_pts = len(saved_train) + len(saved_test)
pt_icon   = "✓" if total_pts == 40 else "✗ FAIL"

cap_display = (
    f"{meta['window_cap']} ({meta['window_cap_mode']})"
    if meta["window_cap"] is not None
    else f"None ({meta['window_cap_mode']})"
)

print(f"\n[{pt_icon}] Total patients processed          : {total_pts}  (expected: 40)")
print(f"[✓] PAF patients in train             : {meta['n_train_paf']}")
print(f"[✓] Non-AF patients in train          : {meta['n_train_nonaf']}")
print(f"[✓] PAF patients in test              : {meta['n_test_paf']}")
print(f"[✓] Non-AF patients in test           : {meta['n_test_nonaf']}")
print(f"[✓] Window cap applied                : {cap_display}")

# Per-patient window counts — TRAIN
print(f"\n[✓] Windows per TRAIN patient:")
print(f"    {'Patient ID':<15} {'Type':<10} {'Windows':>8}")
print(f"    {'-'*15} {'-'*10} {'-'*8}")
train_paf_windows   = 0
train_nonaf_windows = 0
for pid in sorted(saved_train.keys()):
    entry = saved_train[pid]
    print(f"    {pid:<15} {entry['af_type']:<10} {entry['total_windows']:>8}")
    if entry["af_type"] == "PAF":
        train_paf_windows += entry["total_windows"]
    else:
        train_nonaf_windows += entry["total_windows"]

# Per-patient window counts — TEST
print(f"\n[✓] Windows per TEST patient:")
print(f"    {'Patient ID':<15} {'Type':<10} {'Windows':>8}")
print(f"    {'-'*15} {'-'*10} {'-'*8}")
test_paf_windows   = 0
test_nonaf_windows = 0
for pid in sorted(saved_test.keys()):
    entry = saved_test[pid]
    print(f"    {pid:<15} {entry['af_type']:<10} {entry['total_windows']:>8}")
    if entry["af_type"] == "PAF":
        test_paf_windows += entry["total_windows"]
    else:
        test_nonaf_windows += entry["total_windows"]

# Window totals and class balance
total_train = meta["total_train_windows"]
total_test  = meta["total_test_windows"]

tr_paf_pct   = 100 * train_paf_windows   / total_train if total_train > 0 else 0.0
tr_nonaf_pct = 100 * train_nonaf_windows / total_train if total_train > 0 else 0.0
te_paf_pct   = 100 * test_paf_windows    / total_test  if total_test  > 0 else 0.0
te_nonaf_pct = 100 * test_nonaf_windows  / total_test  if total_test  > 0 else 0.0

print(f"\n[✓] Train class balance               : {total_train} total windows")
print(f"    - PAF    : {train_paf_windows:>8}  ({tr_paf_pct:.1f}%)")
print(f"    - non-AF : {train_nonaf_windows:>8}  ({tr_nonaf_pct:.1f}%)")

print(f"[✓] Test class balance                : {total_test} total windows")
print(f"    - PAF    : {test_paf_windows:>8}  ({te_paf_pct:.1f}%)")
print(f"    - non-AF : {test_nonaf_windows:>8}  ({te_nonaf_pct:.1f}%)")

# Cross-check: totals in metadata must match per-patient sums
assert meta["total_train_windows"] == (train_paf_windows + train_nonaf_windows), (
    "Metadata total_train_windows does not match per-patient sum!"
)
assert meta["total_test_windows"] == (test_paf_windows + test_nonaf_windows), (
    "Metadata total_test_windows does not match per-patient sum!"
)

# Patient leakage check (C1)
overlap_ids  = set(saved_train.keys()) & set(saved_test.keys())
leakage_pass = len(overlap_ids) == 0
leak_icon    = "✓" if leakage_pass else "✗ FAIL"
leak_msg     = "PASS" if leakage_pass else f"FAIL — overlapping IDs: {overlap_ids}"
print(f"\n[{leak_icon}] Patient leakage check             : {leak_msg}")

# Temporal order check (C2)
temporal_failures = []
for split_name, split_dict in [("train", saved_train), ("test", saved_test)]:
    for pid, entry in split_dict.items():
        indices = [w["window_index"] for w in entry["windows"]]
        if len(indices) > 1:
            is_ok = all(indices[i] < indices[i + 1] for i in range(len(indices) - 1))
            if not is_ok:
                temporal_failures.append(f"{split_name}/{pid}")

temp_pass = len(temporal_failures) == 0
temp_icon = "✓" if temp_pass else "✗ FAIL"
temp_msg  = "PASS" if temp_pass else f"FAIL — non-monotonic window_index in: {temporal_failures}"
print(f"[{temp_icon}] Temporal order check              : {temp_msg}")

# Recording ID check — every window must belong to the correct recording
rec_failures = []
for split_name, split_dict in [("train", saved_train), ("test", saved_test)]:
    for pid, entry in split_dict.items():
        expected_rec = ALL_PATIENTS[pid]
        wrong = [w["window_index"] for w in entry["windows"]
                 if w["recording_id"] != expected_rec]
        if wrong:
            rec_failures.append(
                f"{split_name}/{pid}: {len(wrong)} window(s) with wrong recording_id "
                f"(expected '{expected_rec}')"
            )

rec_pass = len(rec_failures) == 0
rec_icon = "✓" if rec_pass else "✗ FAIL"
rec_msg  = "PASS" if rec_pass else f"FAIL:\n" + "\n".join(f"  {f}" for f in rec_failures)
print(f"[{rec_icon}] Recording ID check                : {rec_msg}")

print(f"\n[✓] Output file written to            : {CONFIG_STEP_4['output_file']}")

all_pass = (total_pts == 40) and leakage_pass and temp_pass and rec_pass
print("\n" + SEP)
if all_pass:
    print("ALL CHECKS PASSED.")
    print("train_test_split.json is ready for the training pipeline (Script 2).")
else:
    print("WARNING: ONE OR MORE CHECKS FAILED.")
    print("Review the errors flagged above before passing this file to the training pipeline.")
print(SEP)

## Step 4 · Step G — Secondary Test Set Generation (Unselected Patients)

In [ ]:
# =============================================================================
# Step G — Secondary Test Set Generation (Unselected Patients)
# =============================================================================
# Creates a secondary test set from patients not included in the main 40-patient
# selection. These are entirely fresh patients with no data leakage risk.
#
# If a patient has only one valid recording, it is auto-selected.
# If a patient has multiple valid recordings, the user is prompted interactively
# to choose which one to use.
#
# Recordings listed in MANUAL_DISCARD_RECORDINGS (CONFIGURATION section) are
# excluded entirely.
#
# Output: secondary_test_set.json (SECONDARY_TEST_FILE, saved next to the
#         Step 4 train/test split)

def create_secondary_test_set(
    full_pool: dict,
    selected_patients_dict: dict,
    discard_list: list,
    output_path: str
) -> None:
    """
    Create a secondary test set from patients not used in the main Step 4 split.

    Patients whose patient_id appears in selected_patients_dict are skipped
    entirely. For the remaining patients, recordings in the discard_list are
    ignored. If a patient still has multiple valid recordings after filtering,
    the user is prompted interactively to choose one.

    The output mirrors the structure of the Step 4 test dict so it can be
    consumed by the same downstream code without modification.

    Output file: secondary_test_set.json (full path given by output_path)

    Parameters
    ----------
    full_pool              : Complete step3_output.json dict (all patients).
    selected_patients_dict : ALL_PATIENTS dict (patients already used in Step 4).
    discard_list           : List of recording IDs (strings) to exclude entirely.
    output_path            : Full path of the output JSON file.
    """
    print(f"\n{'='*80}")
    print("STEP G - Generating Secondary Test Set (PAF and non-AF)")
    print(f"{'='*80}")

    fresh_patient_selections = {}
    discard_set = set(str(rid) for rid in discard_list)

    # Phase 1: Filter and Select
    for pid, pdata in full_pool.items():
        # Skip patients already used in the main Step 4 split
        if pid in selected_patients_dict:
            continue

        af_type          = pdata.get("af_type", "Unknown")
        all_recs         = set(str(w["recording_id"]) for w in pdata.get("windows", []))
        valid_recs       = [rid for rid in all_recs if rid not in discard_set]

        if len(valid_recs) == 0:
            print(f"  [Skipped] Patient {pid} ({af_type}): All recordings manually discarded.")
            continue

        elif len(valid_recs) == 1:
            chosen_rec = valid_recs[0]
            fresh_patient_selections[pid] = chosen_rec
            print(f"  [Auto-Selected] Patient {pid} ({af_type}): Rec ID [{chosen_rec}] (Only one available).")

        else:
            # Interactive prompt when multiple recordings are available
            sorted_recs   = sorted(valid_recs, key=lambda x: int(x))
            valid_recs_str = ", ".join(sorted_recs)

            print(f"\n  [ACTION REQUIRED] Patient {pid} ({af_type}) has multiple valid recordings: [{valid_recs_str}]")

            while True:
                user_choice = input(
                    f"  >> Which recording do you want to keep? "
                    f"(Example: type '{sorted_recs[0]}' and press Enter): "
                ).strip()

                if user_choice in valid_recs:
                    fresh_patient_selections[pid] = user_choice
                    print(f"  --> Locked in Rec ID [{user_choice}] for Patient {pid}.")
                    break
                else:
                    print(f"  --> INVALID INPUT. You typed '{user_choice}'. "
                          f"Please choose exactly one from: [{valid_recs_str}].")

    # Phase 2: Build the Output Dictionary
    print(f"\n{'-'*80}")
    print("Assembling Data...")

    output_dict        = {}
    total_windows_all  = 0

    for pid, chosen_rec in fresh_patient_selections.items():
        pdata       = full_pool[pid]
        rec_windows = [w for w in pdata["windows"] if str(w["recording_id"]) == chosen_rec]
        total_windows_all += len(rec_windows)

        output_dict[pid] = {
            "af_type"            : pdata["af_type"],
            "recording_id"       : chosen_rec,
            "total_windows"      : len(rec_windows),
            "window_duration_sec": pdata["window_duration_sec"],
            "overlap_pct"        : pdata["overlap_pct"],
            "step_sec"           : pdata["step_sec"],
            "windows"            : rec_windows,
        }

    # Phase 3: Save to JSON
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output_dict, f, indent=2, ensure_ascii=False)

    print(f"{'-'*80}")
    print(f"COMPLETE! Secondary test set created.")
    print(f"  Total Fresh Patients : {len(output_dict)}")
    print(f"  Total Windows        : {total_windows_all:,}")
    print(f"  Saved to             : {output_path}")
    print(f"{'='*80}\n")

## Execute Step G

In [ ]:
create_secondary_test_set(
    full_pool              = pool,
    selected_patients_dict = ALL_PATIENTS,
    discard_list           = MANUAL_DISCARD_RECORDINGS,
    output_path            = SECONDARY_TEST_FILE,
)